<h1> bioRxix and medRxiv: Retrieve data </h1> 

We will be retrieving data from those two preprint servers

<h2></h2>

<h2>bioRxiv</h2>

Since bioRxiv was only founded in 2013, we will be able to fetch articles **starting from 2013**. 

The code below will search the keywords withing **TITLE** and **ABSTRACT** of the article's JSON

In [17]:
import requests, time, pandas as pd

#THESE are keywords by which bioRxiv,medRxiv searches within its articles
PHRASES = [
    "healthy aging","healthy ageing",
    "anti-aging","anti-ageing","anti aging","anti ageing",
    "biohacking","living longer",
    "well aging","well ageing","aging well","ageing well",
    "slow aging","slow ageing",
    "health rejuvenation","nootropics",
]

#This function build a query for search
def build_query(server="bioRxiv", y0=2010, y1=2025, use_synonyms=False):
    or_block      = " OR ".join(f'"{p}"' for p in PHRASES) #joins articles using OR statement, e.g. ("healthy aging" OR "healthy ageing" OR ...)
    q = f'(SRC:PPR) AND ("{server}") AND TITLE_ABS:({or_block}) AND (PUB_YEAR:[{y0} TO {y1}])' #concatenates the query with 
    params = {
        "query": q,
        "format": "json",
        "pageSize": 1000,      # max page size
        "resultType": "core",  # includes abstractText, pmid/pmcid when present
        "cursorMark": "*",     # start of cursor pagination
        "synonym": str(use_synonyms).lower()
    }
    return params

def epmc_search_all(params, pause=0.2, max_pages=None):
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    rows, total, pages = [], None, 0
    while True:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        js = r.json()
        if total is None:
            total = js.get("hitCount")
            print(f"Reported hitCount: {total}") #how many articles found in total
        batch = js.get("resultList", {}).get("result", [])
        rows.extend(batch) # append results to "rows"
        next_cursor = js.get("nextCursorMark")
        if not batch or next_cursor == params["cursorMark"]:
            break
        params["cursorMark"] = next_cursor # paginate through results
        pages += 1
        if max_pages and pages >= max_pages: # if the number of pages is greater than the limit we set, break
            break
        time.sleep(pause) # wait between every fetch
    return rows

if __name__ == "__main__":
    params = build_query(server = "bioRxiv", y0=2010, y1=2025, use_synonyms=False) #HERE you can change "bioRxiv" to "medRxiv" to retreive from 2 different servers
    data   = epmc_search_all(params)
    df     = pd.json_normalize(data)
    
    keep = ["id","pmid","pmcid","doi","title","pubYear","abstractText","source","authorString", "citedByCount", "server"]
    keep = [c for c in keep if c in df.columns]
    df_subset = df.reindex(columns=keep)

    df_subset.to_csv("biorxiv_longevity_titleabs_2010_2025.csv", index=False)
    print(f"Saved {len(df_subset)} rows")

Reported hitCount: 648
Saved 1296 rows


In [18]:
import pandas as pd

df_bio = pd.read_csv("biorxiv_longevity_titleabs_2010_2025.csv")

In [19]:
df_bio.shape

(1296, 8)

In [20]:
df_bio.groupby('citedByCount').size()

citedByCount
0    1088
1     162
2      26
3      16
4       2
7       2
dtype: int64

In [21]:
df_bio['pubYear'].value_counts().sort_index()

pubYear
2015      2
2016     12
2017     34
2018     68
2019     98
2020    144
2021    122
2022    162
2023    244
2024    226
2025    184
Name: count, dtype: int64

<h2> medRxiv</h2>

Since medRxiv was only founded in 2019, we will be able to fetch articles **starting from 2019**. 

The code below will search the keywords withing **TITLE** and **ABSTRACT** of the article's JSON

In [26]:
import requests, time, pandas as pd

#THESE are keywords by which bioRxiv,medRxiv searches within its articles
PHRASES = [
    "longevity", 
    "healthy aging","healthy ageing",
    "anti-aging","anti-ageing","anti aging","anti ageing",
    "biohacking","living longer",
    "well aging","well ageing","aging well","ageing well",
    "slow aging","slow ageing",
    "health rejuvenation","nootropics",
]

#This function build a query for search
def build_query(server="bioRxiv", y0=2010, y1=2025, use_synonyms=False):
    or_block      = " OR ".join(f'"{p}"' for p in PHRASES) #joins articles using OR statement, e.g. ("healthy aging" OR "healthy ageing" OR ...)
    q = f'(SRC:PPR) AND ("{server}") AND TITLE_ABS:({or_block}) AND (PUB_YEAR:[{y0} TO {y1}])' #concatenates the query with 
    params = {
        "query": q,
        "format": "json",
        "pageSize": 1000,      # max page size
        "resultType": "core",  # includes abstractText, pmid/pmcid when present
        "cursorMark": "*",     # start of cursor pagination
        "synonym": str(use_synonyms).lower()
    }
    return params

def epmc_search_all(params, pause=0.2, max_pages=None):
    url = "https://www.ebi.ac.uk/europepmc/webservices/rest/search"
    rows, total, pages = [], None, 0
    while True:
        r = requests.get(url, params=params, timeout=60)
        r.raise_for_status()
        js = r.json()
        if total is None:
            total = js.get("hitCount")
            print(f"Reported hitCount: {total}") #how many articles found in total
        batch = js.get("resultList", {}).get("result", [])
        rows.extend(batch) # append results to "rows"
        next_cursor = js.get("nextCursorMark")
        if not batch or next_cursor == params["cursorMark"]:
            break
        params["cursorMark"] = next_cursor # paginate through results
        pages += 1
        if max_pages and pages >= max_pages: # if the number of pages is greater than the limit we set, break
            break
        time.sleep(pause) # wait between every fetch
    return rows

if __name__ == "__main__":
    params = build_query(server = "medRxiv", y0=2010, y1=2025, use_synonyms=False) #HERE you can change "bioRxiv" to "medRxiv" to retreive from 2 different servers
    data   = epmc_search_all(params)
    df     = pd.json_normalize(data)

    keep = ["id","pmid","pmcid","doi","title","pubYear","abstractText","source","authorString","citedByCount", "server"]
    keep = [c for c in keep if c in df.columns]
    df_subset = df.reindex(columns=keep)

    # #Fill missing citedByCount with zero
    # if "citedByCount" in df_subset.columns:
    #     df_subset["citedByCount"] = df_subset["citedByCount"].fillna(0).astype(int)

    df_subset.to_csv("medrxiv_longevity_titleabs_2010_2025_2.csv", index=False)
    print(f"Saved {len(df_subset)} rows")

Reported hitCount: 428
Saved 856 rows


In [31]:
import json

for record in data[:2]:
    print(json.dumps(record, indent=2))  # pretty print with indentation
    print("="*50)  # separator

{
  "id": "PPR1068272",
  "source": "PPR",
  "doi": "10.1101/2025.08.18.25333703",
  "title": "Evidence-Based Interventions Across the Life Course for Healthy Aging: A Scoping Review made in Colombia",
  "authorString": "Gonz\u00e1lez B LM, Mojica LE, Riveros S, Vanegas Zamora V, G\u00f3mez A, Otero MdP, Chaves F, Jaimes MA, L\u00f3pez V, Ram\u00edrez L, S\u00e1nchez V.",
  "authorList": {
    "author": [
      {
        "fullName": "Gonz\u00e1lez B LM",
        "firstName": "Lina Mar\u00eda",
        "lastName": "Gonz\u00e1lez B",
        "initials": "LM"
      },
      {
        "fullName": "Mojica LE",
        "firstName": "Luis Eduardo",
        "lastName": "Mojica",
        "initials": "LE"
      },
      {
        "fullName": "Riveros S",
        "firstName": "Shannon",
        "lastName": "Riveros",
        "initials": "S"
      },
      {
        "fullName": "Vanegas Zamora V",
        "firstName": "Valentina",
        "lastName": "Vanegas Zamora",
        "initials": "V"
     

In [27]:
import pandas as pd

df_med=pd.read_csv('medrxiv_longevity_titleabs_2010_2025_2.csv')
df_med['pubYear'].value_counts()

pubYear
2024    176
2023    150
2022    140
2025    136
2021    128
2020    116
2019     10
Name: count, dtype: int64

In [28]:
df_med.shape

(856, 8)

In [29]:
df.groupby('citedByCount').size()

citedByCount
0     746
1      76
2      16
3       4
4       2
5       4
8       2
10      2
26      2
52      2
dtype: int64

<h1> bioRxix and medRxiv: Bertopic Analysis </h1> 

<h2> Data preperation </h2> 

We searched the articles first without the keyword "longevity" and with. We will run seperately the bertopic analysis on those data and see if the keyword "longevity" brings more noisy data or not.

In [32]:
import pandas as pd

df_bio = pd.read_csv("biorxiv_longevity_titleabs_2010_2025.csv")
print(df_bio.shape)
df_med = pd.read_csv("medrxiv_longevity_titleabs_2010_2025.csv")
print(df_med.shape)
df_bio_lon = pd.read_csv("biorxiv_longevity_titleabs_2010_2025_2.csv") # data with keyword "longevity"
print(f'Data with longevit keyword: {df_bio_lon.shape}')
df_med_lon = pd.read_csv("medrxiv_longevity_titleabs_2010_2025_2.csv") # data with keyword "longevity"
print(f'Data with longevit keyword: {df_med_lon.shape}')

(1296, 8)
(404, 8)
Data with longevit keyword: (1866, 8)
Data with longevit keyword: (856, 8)


In [33]:
#let's assign server column
df_bio['server'] = 'bioRxiv'
df_med['server'] = 'medRxiv'
df_bio_lon['server'] = 'bioRxiv'
df_med_lon['server'] = 'medRxiv'

In [34]:
#Let's concat the dataframes
import pandas as pd
df = pd.concat([df_bio, df_med], ignore_index=True)
df_lon = pd.concat([df_bio_lon, df_med_lon], ignore_index=True)

In [35]:
#Let's check for the duplicates
print(f"Duplicates without longevity {df['doi'].duplicated().sum()}")
print(f"Duplicates with longevity {df_lon['doi'].duplicated().sum()}")

Duplicates without longevity 852
Duplicates with longevity 434


In [36]:
#droping duplicates
df = df.drop_duplicates(subset=['doi'], keep='first')
df_lon = df_lon.drop_duplicates(subset=['doi'], keep='first')

#check again
print(f"Duplicates without longevity {df['doi'].duplicated().sum()}")
print(f"Duplicates with longevity {df_lon['doi'].duplicated().sum()}")

Duplicates without longevity 0
Duplicates with longevity 0


In [37]:
#Let's check for the null values
print(f'Without longevity: \n{df.isnull().sum()}')
print(f'With longevity: \n{df_lon.isnull().sum()}')

Without longevity: 
id              0
doi             0
title           0
pubYear         0
abstractText    0
source          0
authorString    0
citedByCount    0
server          0
dtype: int64
With longevity: 
id              0
doi             0
title           0
pubYear         0
abstractText    1
source          0
authorString    0
citedByCount    0
server          0
dtype: int64


In [38]:
#let's deal with the null 
df_lon['abstractText'] = df_lon['abstractText'].fillna("")

In [39]:
#Let's check again
print(f'Without longevity: \n{df.isnull().sum()}')
print(f'With longevity: \n{df_lon.isnull().sum()}')

Without longevity: 
id              0
doi             0
title           0
pubYear         0
abstractText    0
source          0
authorString    0
citedByCount    0
server          0
dtype: int64
With longevity: 
id              0
doi             0
title           0
pubYear         0
abstractText    0
source          0
authorString    0
citedByCount    0
server          0
dtype: int64


In [45]:
#Last shape of the data
print(f'Without longevity: {df.shape}')
print(f'With longevity: {df_lon.shape}')

Without longevity: (848, 9)
With longevity: (2288, 9)


We have collected **842 articles** without adding the keyword longevity and **2285 articles** with keyword longevity. Let's save the file and check the topics

In [46]:
#Save data
df.to_csv("../data/clean_biomedRXiv.csv", index=False)
df_lon.to_csv("../data/clean_biomedRXiv_lon.csv", index=False)

<h2> Bertopic run</h2>

<h3> First data (without longevity)</h3>

In [1]:
import pandas as pd
df = pd.read_csv("../data/clean_biomedRXiv.csv")
df.head()

,id,doi,title,pubYear,abstractText,source,authorString,citedByCount,server
0,PPR1019256,10.1101/2025.05.11.651908,Discrimination of Normal from Slow-Aging Mice ...,2025,Tests that can predict whether a drug is likel...,PPR,"Badenoch B, Fiehn O, Rappaport N, Srivastava P...",0,bioRxiv
1,PPR1056170,10.1101/2025.07.25.666736,Nanoalgosomes from <i>Tetraselmis chuii</i>: M...,2025,The search for effective dermocosmetic treatme...,PPR,"Gargano P, Picciotto S, Paterna A, Raccosta S,...",0,bioRxiv
2,PPR1046197,10.1101/2025.06.30.660585,"Reporting quality, effect sizes, and biases fo...",2025,Though interest has grown significantly over t...,PPR,"Parish A, Ioannidis JP, Zhang K, Barardo D, Sw...",0,bioRxiv
3,PPR1047709,10.1101/2025.07.09.663908,Multi-omics characterization of the skin micro...,2025,Shifts in the skin microbiome have shown a clo...,PPR,"Guo D, Chen Y, Wu Y, Cheng J, Lai W, Ma W, Yan...",0,bioRxiv
4,PPR994570,10.1101/2025.03.25.645228,Bisphosphonates Trigger Anti-Ageing Effects Ac...,2025,Bisphosphonates (BPs) have been the major clas...,PPR,"Lu J, Rao SR, Knowles H, Zhan H, Gamez B, Plat...",0,bioRxiv


In [2]:
#Prepare the data
df['text'] = (
    df['title'].fillna('') + ". " + df['abstractText'].fillna('')
).str.lower().str.strip()

docs = df['text'].fillna("").tolist()

In [22]:
#Bertopic
import umap
from hdbscan import HDBSCAN
from bertopic import BERTopic
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Embedding model
model = SentenceTransformer("all-mpnet-base-v2")

# Custom stopwords
custom_stopwords = list(stopwords.words("english")) # + ["which stopwords can i add for biomedarix?"]

# Vectorizer
vectorizer_model = CountVectorizer(
    stop_words=custom_stopwords,
    ngram_range=(1, 2),
    max_df=0.9,   #0.85
    min_df=1,    # 2
    max_features=3000
)

# UMAP
umap_model = umap.UMAP(
    n_neighbors=10,   #15
    n_components=10,  # 5
    min_dist=0.0,
    random_state=42
)

# HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=10,  #20
    min_samples=3,        # 5
    cluster_selection_epsilon=0.0,
    prediction_data=True
)


# BERTopic
topic_model = BERTopic(
    embedding_model=model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    nr_topics=None,
    top_n_words=10,
    calculate_probabilities=True,
)

In [4]:
#Calculate the embeddings
embeddings = model.encode(docs, 
                          batch_size=4, 
                          normalize_embeddings=True, 
                          show_progress_bar=True).astype("float32")

Batches:   0%|          | 0/212 [00:00<?, ?it/s]

In [7]:
import numpy as np
np.save("../data/topics/embeddings.npy", embeddings)

In [9]:
#Run the bertopic
topics, probs = topic_model.fit_transform(docs, embeddings)

In [10]:
#Let's check out the topics
tp = topic_model.get_topic_info()
tp

,Topic,Count,Name,Representation,Representative_Docs
0,-1,196,-1_cognitive_health_older_adults,"[cognitive, health, older, adults, lifespan, g...",[ambient pollution components and sources asso...
1,0,55,0_ad_microglia_brain_alzheimer,"[ad, microglia, brain, alzheimer, plcγ2, neuro...",[trem2-dependent senescent microglia conserved...
2,1,38,1_covid_covid 19_19_pandemic,"[covid, covid 19, 19, pandemic, patients, sars...",[connecting bcg vaccination and covid-19: addi...
3,2,32,2_drosophila_lifespan_fathers_insulin,"[drosophila, lifespan, fathers, insulin, offsp...",[mapping<i>drosophila</i>insulin receptor stru...
4,3,31,3_connectivity_brain_network_functional,"[connectivity, brain, network, functional, net...",[how synchrony and metastable network dynamics...
5,4,28,4_memory_visual_older_episodic,"[memory, visual, older, episodic, neural, adul...",[the vertical position of visual information c...
6,5,23,5_care_social_health_dementia,"[care, social, health, dementia, end life, cog...",[piloting a minimum data set for older people ...
7,6,22,6_mci_mri_asl_slides,"[mci, mri, asl, slides, cognitive, segmentatio...","[multi-cohort, multi-sequence harmonisation fo..."
8,7,22,7_elegans_daf_lifespan_longevity,"[elegans, daf, lifespan, longevity, daf 16, da...",[daf-2/insulin igf-1 receptor regulates motili...
9,8,22,8_childhood_poverty_health_frailty,"[childhood, poverty, health, frailty, course, ...","[growing up in poverty, growing old with multi..."


In [11]:
#create topic column
df['topic'] = topics

<h4>Saving the data</h4>

In [12]:
import numpy as np
import pandas as pd
import pickle


tp.to_csv('../data/topics/topics.csv', index=False)
topic_model.save("../data/topics/topic_model")
np.save('../data/topics/topic_probabilities.npy', probs)
with open('../data/topics/preprocessed_data.pkl', 'wb') as f:
    pickle.dump(docs, f)
df.to_csv('../data/topics/data.csv', index=False)

2025-09-05 14:37:39,352 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [13]:
#Saving without UMAP
topic_model.umap_model = None
topic_model.save("../data/topics/topic_model_no_umap")

2025-09-05 14:38:14,508 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


<h3> Second data (with longevity)</h3>

In [16]:
import pandas as pd
df_lon = pd.read_csv("../data/clean_biomedRXiv_lon.csv")
df_lon.head()

,id,doi,title,pubYear,abstractText,source,authorString,citedByCount,server
0,PPR1019256,10.1101/2025.05.11.651908,Discrimination of Normal from Slow-Aging Mice ...,2025,Tests that can predict whether a drug is likel...,PPR,"Badenoch B, Fiehn O, Rappaport N, Srivastava P...",0,bioRxiv
1,PPR1078166,10.1101/2025.08.29.673072,Evolution of increased longevity and slowed ag...,2025,Evolution has given rise to lifespans in extan...,PPR,"Foley J, McPherson J, Roger M, Batista C, Maux...",0,bioRxiv
2,PPR1056170,10.1101/2025.07.25.666736,Nanoalgosomes from <i>Tetraselmis chuii</i>: M...,2025,The search for effective dermocosmetic treatme...,PPR,"Gargano P, Picciotto S, Paterna A, Raccosta S,...",0,bioRxiv
3,PPR1027860,10.1101/2025.05.26.656117,Methyl Nicotinate Is a Novel Geroprotective Co...,2025,Aging involves cellular decline and reduced st...,PPR,"Mingtong G, Alfatah M.",0,bioRxiv
4,PPR996689,10.1101/2025.03.26.645451,Dietary restriction is evolutionary conserved ...,2025,<h4>ABSTRACT</h4> The anti-ageing response of ...,PPR,"Gautrey SL, Dunning LT, Gossmann TI, Simons MJP.",0,bioRxiv


In [19]:
#Prepare the data
df_lon['text'] = (
    df_lon['title'].fillna('') + ". " + df_lon['abstractText'].fillna('')
).str.lower().str.strip()

docs_lon = df_lon['text'].fillna("").tolist()

In [20]:
#Calculate the embeddings
embeddings = model.encode(docs_lon, 
                          batch_size=4, 
                          normalize_embeddings=True, 
                          show_progress_bar=True).astype("float32")

Batches:   0%|          | 0/572 [00:00<?, ?it/s]

In [21]:
import numpy as np
np.save("../data/topics_lon/embeddings.npy", embeddings)

In [23]:
#Bertopic
import umap
from hdbscan import HDBSCAN
from bertopic import BERTopic
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

# Embedding model
model = SentenceTransformer("all-mpnet-base-v2")

# Custom stopwords
custom_stopwords = list(stopwords.words("english")) # + ["which stopwords can i add for biomedarix?"]

# Vectorizer
vectorizer_model = CountVectorizer(
    stop_words=custom_stopwords,
    ngram_range=(1, 2),
    max_df=0.9,   #0.85
    min_df=1,    # 2
    max_features=3000
)

# UMAP
umap_model = umap.UMAP(
    n_neighbors=10,   #15
    n_components=10,  # 5
    min_dist=0.0,
    random_state=42
)

# HDBSCAN
hdbscan_model = HDBSCAN(
    min_cluster_size=10,  #20
    min_samples=3,        # 5
    cluster_selection_epsilon=0.0,
    prediction_data=True
)


# BERTopic
topic_model = BERTopic(
    embedding_model=model,
    vectorizer_model=vectorizer_model,
    hdbscan_model=hdbscan_model,
    umap_model=umap_model,
    nr_topics=None,
    top_n_words=10,
    calculate_probabilities=True,
)

In [24]:
#Run the bertopic
topics, probs = topic_model.fit_transform(docs_lon, embeddings)

In [34]:
tp_lon = topic_model.get_topic_info()
tp_lon

,Topic,Count,Name,Representation,Representative_Docs
0,-1,406,-1_aging_lifespan_protein_cell,"[aging, lifespan, protein, cell, genes, mice, ...",[transcriptome analysis of the nematode <i>cae...
1,0,317,0_brain_cognitive_adults_older,"[brain, cognitive, adults, older, healthy, mem...",[resting-state eeg aperiodic exponent moderate...
2,1,86,1_biological_dnam_clocks_methylation,"[biological, dnam, clocks, methylation, epigen...",[evaluation of epigenetic and metabolomic biom...
3,2,85,2_genetic_variants_association_risk,"[genetic, variants, association, risk, diabete...",[robust inference and widespread genetic corre...
4,3,69,3_drosophila_dr_flies_lifespan,"[drosophila, dr, flies, lifespan, diet, dietar...",[dietary restriction is evolutionary conserved...
5,4,65,4_cells_cell_cd4_cd8,"[cells, cell, cd4, cd8, immune, hiv, memory, e...",[single-cell transcriptomics reveals expansion...
6,5,54,5_yeast_lifespan_cerevisiae_cells,"[yeast, lifespan, cerevisiae, cells, saccharom...",[rapid nuclear exclusion of hcm1 in aging<i>sa...
7,6,51,6_cancer_mole_genes_evolution,"[cancer, mole, genes, evolution, species, rat,...",[long-lived rodents reveal signatures of posit...
8,7,49,7_microglia_ad_plcγ2_alzheimer,"[microglia, ad, plcγ2, alzheimer, brain, disea...",[protective variant in plcγ2 mitigates alzheim...
9,8,41,8_mitochondrial_atfs_mitochondria_upr mt,"[mitochondrial, atfs, mitochondria, upr mt, up...",[developmental disruption of the mitochondrial...


In [30]:
df_lon['topic']=topics

In [32]:
df_lon.head()

,id,doi,title,pubYear,abstractText,source,authorString,citedByCount,server,text,topic
0,PPR1019256,10.1101/2025.05.11.651908,Discrimination of Normal from Slow-Aging Mice ...,2025,Tests that can predict whether a drug is likel...,PPR,"Badenoch B, Fiehn O, Rappaport N, Srivastava P...",0,bioRxiv,discrimination of normal from slow-aging mice ...,33
1,PPR1078166,10.1101/2025.08.29.673072,Evolution of increased longevity and slowed ag...,2025,Evolution has given rise to lifespans in extan...,PPR,"Foley J, McPherson J, Roger M, Batista C, Maux...",0,bioRxiv,evolution of increased longevity and slowed ag...,-1
2,PPR1056170,10.1101/2025.07.25.666736,Nanoalgosomes from <i>Tetraselmis chuii</i>: M...,2025,The search for effective dermocosmetic treatme...,PPR,"Gargano P, Picciotto S, Paterna A, Raccosta S,...",0,bioRxiv,nanoalgosomes from <i>tetraselmis chuii</i>: m...,36
3,PPR1027860,10.1101/2025.05.26.656117,Methyl Nicotinate Is a Novel Geroprotective Co...,2025,Aging involves cellular decline and reduced st...,PPR,"Mingtong G, Alfatah M.",0,bioRxiv,methyl nicotinate is a novel geroprotective co...,5
4,PPR996689,10.1101/2025.03.26.645451,Dietary restriction is evolutionary conserved ...,2025,<h4>ABSTRACT</h4> The anti-ageing response of ...,PPR,"Gautrey SL, Dunning LT, Gossmann TI, Simons MJP.",0,bioRxiv,dietary restriction is evolutionary conserved ...,3


<h4>Saving the data</h4>

In [35]:
import numpy as np
import pandas as pd
import pickle


tp_lon.to_csv('../data/topics_lon/topics.csv', index=False)
topic_model.save("../data/topics_lon/topic_model")
np.save('../data/topics_lon/topic_probabilities.npy', probs)
with open('../data/topics_lon/preprocessed_data.pkl', 'wb') as f:
    pickle.dump(docs_lon, f)
df_lon.to_csv('../data/topics_lon/data.csv', index=False)

2025-09-05 22:21:33,995 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


In [36]:
#Saving without UMAP
topic_model.umap_model = None
topic_model.save("../data/topics_lon/topic_model_no_umap")

2025-09-05 22:23:04,258 - BERTopic - WARNING: When you use `pickle` to save/load a BERTopic model,please make sure that the environments in which you saveand load the model are **exactly** the same. The version of BERTopic,its dependencies, and python need to remain the same.


<h2> Analyze bertopic </h2>

<h3>Topics with longevity</h3>

Let's first analyze the topics with the word longevity. The "longevity" keyword is the one of the important ones. It will be better to inlude it in our data. That's why we will analyze this data first, and if it will be good, then we will use it.

In [5]:
#To show max amount of vidoes
pd.set_option("display.max_rows", None)

In [6]:
import pandas as pd

tp = pd.read_csv("../data/topics_lon/topics.csv")
df = pd.read_csv("../data/topics_lon/data.csv")

Let's check the each topic individually to see whether it is related or not. 

In [7]:
tp

,Topic,Count,Name,Representation,Representative_Docs
0,-1,406,-1_aging_lifespan_protein_cell,"['aging', 'lifespan', 'protein', 'cell', 'gene...",['transcriptome analysis of the nematode <i>ca...
1,0,317,0_brain_cognitive_adults_older,"['brain', 'cognitive', 'adults', 'older', 'hea...",['resting-state eeg aperiodic exponent moderat...
2,1,86,1_biological_dnam_clocks_methylation,"['biological', 'dnam', 'clocks', 'methylation'...",['evaluation of epigenetic and metabolomic bio...
3,2,85,2_genetic_variants_association_risk,"['genetic', 'variants', 'association', 'risk',...",['robust inference and widespread genetic corr...
4,3,69,3_drosophila_dr_flies_lifespan,"['drosophila', 'dr', 'flies', 'lifespan', 'die...",['dietary restriction is evolutionary conserve...
5,4,65,4_cells_cell_cd4_cd8,"['cells', 'cell', 'cd4', 'cd8', 'immune', 'hiv...",['single-cell transcriptomics reveals expansio...
6,5,54,5_yeast_lifespan_cerevisiae_cells,"['yeast', 'lifespan', 'cerevisiae', 'cells', '...",['rapid nuclear exclusion of hcm1 in aging<i>s...
7,6,51,6_cancer_mole_genes_evolution,"['cancer', 'mole', 'genes', 'evolution', 'spec...",['long-lived rodents reveal signatures of posi...
8,7,49,7_microglia_ad_plcγ2_alzheimer,"['microglia', 'ad', 'plcγ2', 'alzheimer', 'bra...",['protective variant in plcγ2 mitigates alzhei...
9,8,41,8_mitochondrial_atfs_mitochondria_upr mt,"['mitochondrial', 'atfs', 'mitochondria', 'upr...",['developmental disruption of the mitochondria...


In [10]:
#to show the columns
pd.set_option("display.max_colwidth", None)

In [32]:
df['is_relevant']=True
tp['is_relevant']=True

In [112]:
#check topics
dfp = df[df['topic']==65].reset_index()
tpp = tp[tp['Topic']==65]
print(f"Keywords: {tpp['Representation'].head()}")
for i in range(0, 6):
    print(f"\n--- Title {i} ---\n{dfp.iloc[i]['title']}")
    print(f"--- Abstract {i} ---\n{dfp.iloc[i]['abstractText']}")

Keywords: 66    ['tdp 43', '43', 'tdp', 'pathology', 'neurons', 'behavioural', 'repeat', 'mice', 'th', 'progressive']
Name: Representation, dtype: object

--- Title 0 ---
Artemin sensitises mouse (<i>Mus musculus</i>) and naked mole-rat (<i>Heterocephalus glaber</i>) sensory neurons <i>in vitro</i>
--- Abstract 0 ---
The naked mole-rat (NMR, Heterocephalus glaber ) is a subterranean rodent that exhibits a range of unusual physiological traits, including diminished inflammatory pain. For example, nerve growth factor (NGF), a key inflammatory mediator, fails to induce sensitization of sensory neurons and thermal hyperalgesia in NMRs. This lack of NGF-induced neuronal sensitization and thermal hyperalgesia results from hypofunctional signaling of the NGF receptor, tropomyosin receptor kinase A (TrkA). Like NGF-TrkA signaling, the neurotrophic factor artemin, a member of the glial cell line-derived neurotrophic factor (GDNF) family, is implicated in mediating inflammatory pain through its 

In [103]:
dfp.shape

(12, 13)

- The topic 6 is about mostly the longevity of the animals. Should be deleted or not?
- The topic 9 is about the pests and mosquitos
- The topic 10 is about seed longevity. Should be deleted or not?
- Topic 11: anti - aging and longevity mechanism in bees
- Topic 16: about vaccines. Should be deleted or not?
- Topic 17: about mating pests
- Topic 19: about globalization and population of the species
- Topic 20: About antibiotics (not related to longevity or health)
- Topic 21: ? not related
- Topic 22: delete
- Topic 25: About the longevity in the trees
- Topic 26: Delete:
- Topic 31: focused exclusively on the SARS-CoV-2 virus and the pandemic response
- Topic 52: only talks about o SARS-CoV-2 (COVID-19) and other viruses, without talking linking it with longevity
- Topic 53: focused on the practical public health and clinical aspects of the pandemic, not the underlying biological links to the aging process.
- Topic 54: focused on vaccine development and the immune response to infectious diseases, specifically malaria and, in one case, Zika virus.
- Topic 59: bats as a model organism for exceptional longevity
- Topic 62: focused on botany, agriculture, and plant pathology.
- Topic 63: focused on entomology and microbiology

In [114]:
#List of unrelated topics
u_list = [6, 9, 10, 11, 16, 17, 19, 20, 21, 22, 25, 26, 31, 52, 53, 54, 59, 62, 63]

In [115]:
#False if not relevant
for i in u_list:
    df.loc[df['topic']==i, 'is_relevant']=False
    tp.loc[tp['Topic']==i, 'is_relevant']=False

In [117]:
tp.to_csv("../data/topics_lon/final_topics.csv", index=False)
df.to_csv("../data/topics_lon/final_data.csv", index=False)

In [118]:
df.groupby('is_relevant').size()

is_relevant
False     450
True     1838
dtype: int64

Out of of 2288 articles, 1838 are analyzed as related